In [ ]:
def repeat_bit_sequence1(b, l, n):
    result = 0
    for i in range(n):
        result |= b
        b <<= l
    return result

In [ ]:
print(bin(repeat_bit_sequence1(b=0b11001, l=5, n=5)))

0b1100111001110011100111001


In [ ]:
def repeat_bit_sequence2(b, l, n):
    return int(f"{b:0{l}b}" * n, 2)

In [ ]:
print(bin(repeat_bit_sequence2(b=0b11001, l=5, n=5)))

0b1100111001110011100111001


In [ ]:
def repeat_bit_sequence3(b, l, n):
    r = 1 << l
    return b * ((1 - (r ** n)) // (1 - r))

In [ ]:
print(bin(repeat_bit_sequence3(b=0b11001, l=5, n=5)))

0b1100111001110011100111001


In [ ]:
for n in [10, 100, 1000, 10000, 100000]:
    print(f"n = {n}")
    %timeit repeat_bit_sequence1(b=0b11001, l=5, n=n)
    %timeit repeat_bit_sequence2(b=0b11001, l=5, n=n)
    %timeit repeat_bit_sequence3(b=0b11001, l=5, n=n)

n = 10
1.03 μs ± 13.4 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
674 ns ± 9.33 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
345 ns ± 1.89 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
n = 100
10.5 μs ± 71.9 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
1.5 μs ± 2.41 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
737 ns ± 3.74 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)
n = 1000
231 μs ± 9.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
9.54 μs ± 34.9 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
6.99 μs ± 68.5 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
n = 10000
10.9 ms ± 21.7 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
81.2 μs ± 117 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
105 μs ± 250 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
n = 100000
955 ms ± 4 ms per loop (mean ± std. dev. of 7 runs

In [ ]:
from marubatsu import BitBoard

def __init__(self, board_size:int=3, count_linemark:bool=False):
    self.BOARD_SIZE = board_size
    self.bit_length = self.BOARD_SIZE ** 2
    self.count_linemark = count_linemark

    # 参照テーブルの計算
    self.fullmask = (1 << self.BOARD_SIZE ** 2) - 1
    self.colmasks = []
    self.rowmasks = []
    self.diamask1 = 0
    self.diamask2 = 0
    for i in range(self.BOARD_SIZE):
        colmask = 0
        rowmask = 0
        for j in range(self.BOARD_SIZE):
            colmask |= self.xy_to_move(i, j)
            rowmask |= self.xy_to_move(j, i)
        self.colmasks.append(colmask)
        self.rowmasks.append(rowmask)
        self.diamask1 |= self.xy_to_move(i, i)
        self.diamask2 |= self.xy_to_move(i, self.BOARD_SIZE - i - 1)
    self.masklist = self.colmasks + self.rowmasks + [self.diamask1, self.diamask2]
    
    self.fliplr_ds_table = []
    mask = None
    length = self.BOARD_SIZE
    while length > 1:
        delta = (length + 1) // 2 * self.BOARD_SIZE
        length //= 2
        if mask is None:
            mask = (1 << (length * self.BOARD_SIZE)) - 1
        else:
            m = mask & (mask >> delta)
            mask = m | (m << prevdelta)
        self.fliplr_ds_table.append((delta, mask))
        prevdelta = delta
        
    self.BB_SIZE = 1 << (self.BOARD_SIZE - 1).bit_length()
    self.delta = (self.BB_SIZE - self.BOARD_SIZE) * self.BOARD_SIZE
    self.fliplr_sa_table = []
    mask = None
    length = self.BB_SIZE
    while length > 1:
        length //= 2
        delta = length * self.BOARD_SIZE
        if mask is None:
            mask = (1 << (length * self.BOARD_SIZE)) - 1
        else:
            m = mask & (mask >> delta)
            mask = m | (m << prevdelta)
        self.fliplr_sa_table.append((delta, mask))
        prevdelta = delta

    self.transpose_ds_table = []
    for i in range(1, self.BOARD_SIZE):
        mask = 0
        for j in range(self.BOARD_SIZE - i):
            mask |= self.xy_to_move(j, i + j)
        self.transpose_ds_table.append(((self.BOARD_SIZE - 1) * i, mask))
            

    def repeat_bit_sequence(b, l, n):
        r = 1 << l
        return b * ((1 - (r ** n)) // (1 - r))

    self.transpose_dc_table = []
    n = self.BOARD_SIZE
    ni = n
    while ni > 1:
        ni //= 2
        delta = (n - 1) * ni
        # M1 を計算する
        mask = ((1 << ni) - 1) << ni
        # M3 を計算する
        mask = repeat_bit_sequence(mask, ni * 2, self.BOARD_SIZE // 2)
        # ビットマスクを計算する        
        mask = repeat_bit_sequence(mask, self.BOARD_SIZE * ni * 2, self.BOARD_SIZE // (ni * 2))
        self.transpose_dc_table.append((delta, mask))
            
    self.initialize()
    
BitBoard.__init__ = __init__

In [9]:
from marubatsu import Marubatsu

mb8 = Marubatsu(boardclass=BitBoard, board_size=8)
for x in range(8):
    for y in range(x, 8):
        mb8.cmove(x, y)
print(mb8)
mb8.board.transpose_dc()
print(mb8)
mb8.board.transpose_dc()
print(mb8)

Turn o
o.......
xo......
oxx.....
xoox....
oxxoo...
xooxxo..
oxxooxx.
xooxxooX

Turn o
oxoxoxox
.oxoxoxo
..xoxoxo
...xoxox
....oxox
.....oxo
......xo
.......X

Turn o
o.......
xo......
oxx.....
xoox....
oxxoo...
xooxxo..
oxxooxx.
xooxxooX



In [ ]:
def __init__(self, board_size=3, count_linemark=False):
    self.BOARD_SIZE = board_size
    self.BB_SIZE = 1 << (self.BOARD_SIZE - 1).bit_length() 
    self.bit_length = self.BB_SIZE ** 2
    self.square_num = self.BOARD_SIZE ** 2
    self.count_linemark = count_linemark

    # 参照テーブルの計算
    self.fullmask = 0
    self.colmasks = []
    self.rowmasks = []
    self.diamask1 = 0
    self.diamask2 = 0
    for i in range(self.BOARD_SIZE):
        colmask = 0
        rowmask = 0
        for j in range(self.BOARD_SIZE):
            colmask |= self.xy_to_move(i, j)
            rowmask |= self.xy_to_move(j, i)
            self.fullmask |= self.xy_to_move(i, j)
        self.colmasks.append(colmask)
        self.rowmasks.append(rowmask)
        self.diamask1 |= self.xy_to_move(i, i)
        self.diamask2 |= self.xy_to_move(i, self.BOARD_SIZE - i - 1)
    self.masklist = self.colmasks + self.rowmasks + [self.diamask1, self.diamask2]
    
    self.fliplr_ds_table = []
    mask = None
    length = self.BOARD_SIZE
    while length > 1:
        delta = (length + 1) // 2 * self.BB_SIZE
        length //= 2
        if mask is None:
            mask = (1 << (length * self.BB_SIZE)) - 1
        else:
            m = mask & (mask >> delta)
            mask = m | (m << prevdelta)
        self.fliplr_ds_table.append((delta, mask))
        prevdelta = delta
        
    self.delta = (self.BB_SIZE - self.BOARD_SIZE) * self.BB_SIZE
    self.fliplr_sa_table = []
    mask = None
    length = self.BB_SIZE
    while length > 1:
        length //= 2
        delta = length * self.BB_SIZE
        if mask is None:
            mask = (1 << (length * self.BB_SIZE)) - 1
        else:
            m = mask & (mask >> delta)
            mask = m | (m << prevdelta)
        self.fliplr_sa_table.append((delta, mask))
        prevdelta = delta

    self.transpose_ds_table = []
    for i in range(1, self.BOARD_SIZE):
        mask = 0
        for j in range(self.BOARD_SIZE - i):
            mask |= self.xy_to_move(j, i + j)
        self.transpose_ds_table.append(((self.BB_SIZE - 1) * i, mask))           

    def repeat_bit_sequence(b, l, n):
        r = 1 << l
        return b * ((1 - (r ** n)) // (1 - r))

    self.transpose_dc_table = []
    n = self.BB_SIZE
    ni = n
    while ni > 1:
        ni //= 2
        delta = (n - 1) * ni
        # M1 を計算する
        mask = ((1 << ni) - 1) << ni
        # M3 を計算する
        mask = repeat_bit_sequence(mask, ni * 2, self.BB_SIZE // 2)
        # ビットマスクを計算する        
        mask = repeat_bit_sequence(mask, self.BB_SIZE * ni * 2, self.BB_SIZE // (ni * 2))
        self.transpose_dc_table.append((delta, mask))
            
    self.initialize()
    
BitBoard.__init__ = __init__

In [11]:
def xy_to_move(self, x, y):
    return 1 << (y + self.BB_SIZE * x)
    
BitBoard.xy_to_move = xy_to_move

def move_to_xy(self, move):
    move = move.bit_length() - 1
    return move // self.BB_SIZE, move % self.BB_SIZE   

BitBoard.move_to_xy = move_to_xy

In [12]:
def judge(self, last_turn, last_move, move_count):
    if move_count < self.BOARD_SIZE * 2 - 1:
        return Marubatsu.PLAYING
    # 直前に着手を行ったプレイヤーの勝利の判定
    if self.is_winner(last_turn, last_move):
        return last_turn
    # 引き分けの判定
    elif move_count == self.square_num:
        return Marubatsu.DRAW
    # 上記のどれでもなければ決着がついていない
    else:
        return Marubatsu.PLAYING 
    
BitBoard.judge = judge

In [13]:
def calc_same_hashables(self, move=None):
    if move is None:
        hashables = set()
    else:
        hashables = {}
    if move is not None:
        x, y = self.move_to_xy(move)
    
    for _ in range(4):
        # 左右の反転処理
        self.fliplr_sa()
        hashable = self.board_to_hashable()
        if move is None:
            hashables.add(hashable)
        else:
            x = self.BOARD_SIZE - x - 1
            hashables[hashable] = self.xy_to_move(x, y)
            
        # 転置処理
        self.transpose_dc()
        hashable = self.board_to_hashable()
        if move is None:
            hashables.add(hashable)
        else:
            x, y = y, x
            hashables[hashable] = self.xy_to_move(x, y)

    return hashables  

BitBoard.calc_same_hashables = calc_same_hashables

In [14]:
mb = Marubatsu(boardclass=BitBoard, board_size=5)
for x in range(5):
    for y in range(x, 5):
        mb.cmove(x, y)
print(mb)
print()
print("fliplr_ds")
mb.board.fliplr_ds()
print(mb)
mb.board.fliplr_ds()
print(mb)
print()

print("fliplr_sa")
mb.board.fliplr_sa()
print(mb)
mb.board.fliplr_sa()
print(mb)
print()

print("transpose_ds")
mb.board.transpose_ds()
print(mb)
mb.board.transpose_ds()
print(mb)
print()

print("transpose_dc")
mb.board.transpose_dc()
print(mb)
mb.board.transpose_dc()
print(mb)
print()

Turn x
o....
xx...
oox..
xxoo.
ooxxO


fliplr_ds
Turn x
....o
...xx
..xoo
.ooxx
oxxoO

Turn x
o....
xx...
oox..
xxoo.
ooxxO


fliplr_sa
Turn x
....o
...xx
..xoo
.ooxx
oxxoO

Turn x
o....
xx...
oox..
xxoo.
ooxxO


transpose_ds
Turn x
oxoxo
.xoxo
..xox
...ox
....O

Turn x
o....
xx...
oox..
xxoo.
ooxxO


transpose_dc
Turn x
oxoxo
.xoxo
..xox
...ox
....O

Turn x
o....
xx...
oox..
xxoo.
ooxxO




In [15]:
mb = Marubatsu(boardclass=BitBoard)
mb.cmove(1, 1)
mb.cmove(0, 0)
mb.cmove(1, 0)
move = mb.board.xy_to_move(1, 0)
hashables = mb.board.calc_same_hashables(move)
for hashable, move in hashables.items():
    mb.board.board[0] = hashable & mb.board.fullmask
    mb.board.board[1] = hashable >> mb.board.bit_length
    print(mb.board.move_to_xy(move))
    print(mb)
    print()

(1, 0)
Turn x
.Ox
.o.
...


(0, 1)
Turn x
...
oo.
x..


(2, 1)
Turn x
...
.oo
..x


(1, 2)
Turn x
...
.o.
.ox


(1, 2)
Turn x
...
.o.
xo.


(2, 1)
Turn x
..x
.oo
...


(0, 1)
Turn x
x..
oo.
...


(1, 0)
Turn x
xO.
.o.
...




In [16]:
from util import benchmark

benchmark(mbparams={"boardclass": BitBoard})

ai2 VS ai2


100%|██████████| 50000/50000 [00:02<00:00, 22846.27it/s]


count     win    lose    draw
o       29454   14352    6194
x       14208   29592    6200
total   43662   43944   12394

ratio     win    lose    draw
o       58.9%   28.7%   12.4%
x       28.4%   59.2%   12.4%
total   43.7%   43.9%   12.4%

ai14s VS ai2


100%|██████████| 50000/50000 [00:15<00:00, 3168.89it/s]


count     win    lose    draw
o       49446       0     554
x       44043       0    5957
total   93489       0    6511

ratio     win    lose    draw
o       98.9%    0.0%    1.1%
x       88.1%    0.0%   11.9%
total   93.5%    0.0%    6.5%

ai_abs_dls
 12.4 ms ±   1.2 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [17]:
from marubatsu import BitBoard3x3

def xy_to_move(self, x:int, y:int) -> int:
    return 1 << (y + self.BOARD_SIZE * x)

BitBoard3x3.xy_to_move = xy_to_move
        
def move_to_xy(self, move:int) -> tuple[int, int]:
    move = move.bit_length() - 1
    return move // self.BOARD_SIZE, move % self.BOARD_SIZE  

BitBoard3x3.move_to_xy = move_to_xy

In [18]:
benchmark(mbparams={"boardclass":BitBoard3x3})

ai2 VS ai2


100%|██████████| 50000/50000 [00:02<00:00, 24419.94it/s]


count     win    lose    draw
o       29454   14352    6194
x       14208   29592    6200
total   43662   43944   12394

ratio     win    lose    draw
o       58.9%   28.7%   12.4%
x       28.4%   59.2%   12.4%
total   43.7%   43.9%   12.4%

ai14s VS ai2


100%|██████████| 50000/50000 [00:15<00:00, 3313.22it/s]


count     win    lose    draw
o       49446       0     554
x       44043       0    5957
total   93489       0    6511

ratio     win    lose    draw
o       98.9%    0.0%    1.1%
x       88.1%    0.0%   11.9%
total   93.5%    0.0%    6.5%

ai_abs_dls
  8.4 ms ±   1.6 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [ ]:
from marubatsu import ListBoard, List1dBoard, ArrayBoard, NpBoard, \
                      NpIntBoard, NpBoolBoard

for board_size in [3, 5, 10, 50, 100]:
    print(f"board size = {board_size}")
    for boardclass in [ListBoard, List1dBoard, ArrayBoard, NpBoard, NpIntBoard, 
                       NpBoolBoard, BitBoard, BitBoard3x3]:
        if board_size != 3 and boardclass == BitBoard3x3:
            continue
        print(f"boardclass: {boardclass.__name__}")
        mb = Marubatsu(boardclass=boardclass, board_size=board_size)
        for i in range(board_size - 1):
            mb.cmove(i, 0)
            mb.cmove(i, 1)
        %timeit mb.board.calc_same_hashables()
        print()

board size = 3
boardclass: ListBoard
21.1 μs ± 1.48 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)

boardclass: List1dBoard
7.39 μs ± 31.4 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)

boardclass: ArrayBoard
7.98 μs ± 109 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)

boardclass: NpBoard
50.9 μs ± 112 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)

boardclass: NpIntBoard
51.1 μs ± 1.04 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)

boardclass: NpBoolBoard
55.7 μs ± 931 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)

boardclass: BitBoard
10.6 μs ± 75.5 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)

boardclass: BitBoard3x3
4.31 μs ± 20.9 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)

board size = 5
boardclass: ListBoard
46.9 μs ± 944 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)

boardclass: List1dBoard
11.1 μs ± 18.3 ns per loop (mean ± std. dev. of 7 runs, 100